# M05-02 — Acumulados por entidad

[← Anterior](02-lab-ranking-ventana.ipynb) · [Siguiente →](../M06-optimizacion-ejecucion/01-teoria.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Numerar los pedidos de cada cliente (`order_n`) y el GMV cobrable **acumulado** en el tiempo (`gmv_running`).

Misma ventana que la teoría: `partitionBy(cliente)` + `orderBy(fecha)`. Si `gmv_running` baja dentro de un cliente, el orden está mal — no lo copies: **compruébalo**.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M05-02-acumulados.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Primero: grano pedido (no línea)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Un pedido con 3 productos no es 3 visitas. Si rankeas o acumulas el fact a palo seco, `order_n` cuenta **líneas**.

Por eso agrupas a `order_id`: fecha del pedido = `min(order_ts)`, dinero = `sum(gmv_line)`.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** **469** (el mismo count que el KPI de M04-02). Si salen **1122**, no agregaste a `order_id`: estás en grano línea.

**Por qué este paso.** 469 tickets ≠ 1122 líneas. El acumulado “por visita” vive en el ticket.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, min as fmin, sum as fsum, row_number

spark = get_spark("novashop-m05")
orders_gmv = (
    spark.read.parquet(str(STAGING / "fact_lines"))
    .join(spark.read.parquet(str(STAGING / "customers_clean")), "customer_id", "inner")
    .where(col("is_billable"))
    .groupBy("customer_id", "order_id")
    .agg(
        fmin("order_ts").alias("order_ts"),  # un instante por ticket
        fsum("gmv_line").alias("gmv"),       # dinero de todas las líneas del ticket
    )
)
print("pedidos cobrables con cliente", orders_gmv.count())


## Prueba tú — ¿Qué pasa si no agrupas?

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

Cuenta el fact cobrable+inner **sin** el `groupBy` de `order_id`. Compáralo con 469.

**Qué tienes que ver.** `líneas` **1122**, `pedidos distintos` **469**. Si usas 1122 como “nº de pedido”, estás inflando visitas.


In [ ]:
lineas = (
    spark.read.parquet(str(STAGING / "fact_lines"))
    .join(spark.read.parquet(str(STAGING / "customers_clean")), "customer_id", "inner")
    .where(col("is_billable"))
)
print("líneas", lineas.count(), "pedidos distintos", lineas.select("order_id").distinct().count())


### Paso 2 — Número de pedido y acumulado

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Una sola window para las dos columnas: el vecindario es el cliente; el eje es el tiempo.

`row_number` → 1.er, 2.º, 3.er ticket de **esa** persona.
`sum(gmv).over(w)` → dinero desde el primer ticket **hasta este** (incluido).

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `order_n` 1, 2, 3… **por cliente**. `gmv_running` no decrece dentro del mismo `customer_id`.

**Por qué este paso.** Si baja, el `orderBy` de la window no es `order_ts` (o está descendente).


In [ ]:
from pyspark.sql.window import Window

w = Window.partitionBy("customer_id").orderBy("order_ts")
hist = (
    orders_gmv.withColumn("order_n", row_number().over(w))
    .withColumn("gmv_running", fsum("gmv").over(w))
)
hist.orderBy("customer_id", "order_n").show(12)


## Prueba tú — Un cliente concreto

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

Elige un `customer_id` que en el `show` tenga `order_n` ≥ 2. Filtra solo ese id y mira si la fila 2 tiene `gmv_running` ≥ fila 1. Anota id y las dos filas en Markdown.

**Qué tienes que ver.** Al menos dos filas. `gmv_running` de `order_n=2` ≥ el de `order_n=1`. Si no, el orden de la ventana está al revés.


In [ ]:
# Cambia el id por uno que hayas visto con varios pedidos
cid = hist.where(col("order_n") >= 2).select("customer_id").first()["customer_id"]
print("cliente", cid)
hist.where(col("customer_id") == cid).orderBy("order_n").show()


### Paso 3 — Primera compra vs repetición

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

`order_n == 1` es la definición de “nuevo” en este curso: primer ticket cobrable de ese cliente. El resto son repeticiones.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** ~211 primeras compras (`true`: un cliente con paid) y el resto `false` (repeticiones). 211 + repeticiones = 469.

**Por qué este paso.** Sin `partitionBy`, `order_n=1` sería **una sola fila en toda la empresa**.


In [ ]:
hist.groupBy((col("order_n") == 1).alias("is_first")).count().show()


## Prueba tú — Ventana de toda la empresa

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

Repite el paso 2 con `w_emp = Window.orderBy("order_ts")` (sin partitionBy). Cuenta cuántos `order_n == 1` hay.

**Qué tienes que ver.** Sin `partitionBy`: **1**. Con él: ~**211**. Esa diferencia *es* la ventana.


In [ ]:
w_emp = Window.orderBy("order_ts")  # un solo ranking temporal global
hist_emp = orders_gmv.withColumn("order_n", row_number().over(w_emp))
print("order_n=1 sin partitionBy", hist_emp.where(col("order_n") == 1).count())  # 1
print("order_n=1 con partitionBy", hist.where(col("order_n") == 1).count())     # ~211


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Un cliente con `order_n` ≥ 2: `gmv_running` fila 2 ≥ fila 1. Markdown con el id.

Counts: 469 pedidos; ~211 primeros; `order_n=1` global (sin partitionBy) = 1.


## Mejora — Pedidos hasta superar 1000 €

Quédate, por cliente, con la **primera** fila donde `gmv_running >= 1000` (o ninguna si no llega). Markdown: ¿`order_n` 1 o hace falta el 2.º ticket?

Si te atasca, el código está en la celda siguiente.


In [ ]:
w2 = Window.partitionBy("customer_id").orderBy("order_ts")
crossed = hist.where(col("gmv_running") >= 1000)
first_cross = crossed.withColumn("rn", row_number().over(w2)).where(col("rn") == 1)
first_cross.select("customer_id", "order_n", "gmv_running").show()
print("clientes que cruzan 1000", first_cross.count())


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| gmv_running igual en todas las filas | Window sin orderBy | partitionBy + orderBy(order_ts) |
| 1122 “pedidos” | No agregaste a order_id | Paso 1 |
| Acumulado a nivel empresa | Falta partitionBy | Añádelo |
| order_n=1 solo una vez | Ventana global | partitionBy(customer_id) |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M06 — teoría](../M06-optimizacion-ejecucion/01-teoria.ipynb).
